# Step 1

In [1]:
import pandas as pd
import numpy as np

from itertools import combinations

### 1 Load Data

In [2]:
ratings = pd.read_csv(
    "../data/1mil/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "../data/1mil/movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

users = pd.read_csv(
    "../data/1mil/users.dat",
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)

### 2 Data Exploration

In [3]:
print("SHAPES")
print("ratings:", ratings.shape)
print("movies:", movies.shape)
print("users:", users.shape)

print("\nNaN check")
print("ratings:\n", ratings.isnull().sum())
print("\nmovies:\n", movies.isnull().sum())
print("\nusers:\n", users.isnull().sum())

print("\nDuplicate check")
print("Duplicate rows in ratings:", ratings.duplicated().sum())
print("Duplicate rows in movies:", movies.duplicated().sum())
print("Duplicate rows in users:", users.duplicated().sum())

print("\nRating values")
print(sorted(ratings["rating"].unique()))

print("\nConsistency check")
print("All user_id in ratings exist in users:",
      ratings["user_id"].isin(users["user_id"]).all())
print("All movie_id in ratings exist in movies:",
      ratings["movie_id"].isin(movies["movie_id"]).all())

SHAPES
ratings: (1000209, 4)
movies: (3883, 3)
users: (6040, 5)

NaN check
ratings:
 user_id      0
movie_id     0
rating       0
timestamp    0
dtype: int64

movies:
 movie_id    0
title       0
genres      0
dtype: int64

users:
 user_id       0
gender        0
age           0
occupation    0
zip_code      0
dtype: int64

Duplicate check
Duplicate rows in ratings: 0
Duplicate rows in movies: 0
Duplicate rows in users: 0

Rating values
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Consistency check
All user_id in ratings exist in users: True
All movie_id in ratings exist in movies: True


In [4]:
print("Ratings sample")
display(ratings.head())

print("Movies sample")
display(movies.head())

print("Users sample")
display(users.head())

print("Rating distribution")
print(ratings["rating"].value_counts().sort_index())

print("\nUnique users in ratings:", ratings["user_id"].nunique())
print("Unique movies in ratings:", ratings["movie_id"].nunique())

Ratings sample


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


Movies sample


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


Users sample


,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


Rating distribution
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

Unique users in ratings: 6040
Unique movies in ratings: 3706


In [5]:
user_counts = ratings.groupby("user_id")["movie_id"].count()
movie_counts = ratings.groupby("movie_id")["user_id"].count()

print("User rating count summary")
print(user_counts.describe())

print("\nMovie rating count summary")
print(movie_counts.describe())

User rating count summary
count    6040.000000
mean      165.597517
std       192.747029
min        20.000000
25%        44.000000
50%        96.000000
75%       208.000000
max      2314.000000
Name: movie_id, dtype: float64

Movie rating count summary
count    3706.000000
mean      269.889099
std       384.047838
min         1.000000
25%        33.000000
50%       123.500000
75%       350.000000
max      3428.000000
Name: user_id, dtype: float64


In [6]:
MIN_USER_RATINGS = 20
MIN_MOVIE_RATINGS = 20

print("Users with >= 20 ratings:", (user_counts >= MIN_USER_RATINGS).sum())
print("Movies with >= 20 ratings:", (movie_counts >= MIN_MOVIE_RATINGS).sum())

Users with >= 20 ratings: 6040
Movies with >= 20 ratings: 3043


### 3 Filtering

In [7]:
def filter_users_movies(df, min_user_ratings=20, min_movie_ratings=20):
    filtered = df.copy()

    while True:
        n_before = len(filtered)

        valid_users = filtered["user_id"].value_counts()
        valid_users = valid_users[valid_users >= min_user_ratings].index
        filtered = filtered[filtered["user_id"].isin(valid_users)]

        valid_movies = filtered["movie_id"].value_counts()
        valid_movies = valid_movies[valid_movies >= min_movie_ratings].index
        filtered = filtered[filtered["movie_id"].isin(valid_movies)]

        n_after = len(filtered)

        if n_before == n_after:
            break

    return filtered


filtered_ratings = filter_users_movies(
    ratings,
    min_user_ratings=20,
    min_movie_ratings=20
)

print("Original ratings shape:", ratings.shape)
print("Filtered ratings shape:", filtered_ratings.shape)
print("Filtered unique users:", filtered_ratings["user_id"].nunique())
print("Filtered unique movies:", filtered_ratings["movie_id"].nunique())

Original ratings shape: (1000209, 4)
Filtered ratings shape: (995154, 4)
Filtered unique users: 6022
Filtered unique movies: 3043


In [8]:
print("Min ratings per user after filtering:",
      filtered_ratings["user_id"].value_counts().min())

print("Min ratings per movie after filtering:",
      filtered_ratings["movie_id"].value_counts().min())

Min ratings per user after filtering: 20
Min ratings per movie after filtering: 20


In [9]:
filtered_ratings = filtered_ratings.copy()
filtered_ratings["datetime"] = pd.to_datetime(filtered_ratings["timestamp"], unit="s")

In [10]:
print("Time range:")
print(filtered_ratings["datetime"].min(), "-", filtered_ratings["datetime"].max())
filtered_ratings.head()

Time range:
2000-04-25 23:05:32 - 2003-02-28 17:49:50


,user_id,movie_id,rating,timestamp,datetime
0,1,1193,5,978300760,2000-12-31 22:12:40
1,1,661,3,978302109,2000-12-31 22:35:09
2,1,914,3,978301968,2000-12-31 22:32:48
3,1,3408,4,978300275,2000-12-31 22:04:35
4,1,2355,5,978824291,2001-01-06 23:38:11


### 4 Split

In [11]:
def temporal_split(df, val_ratio=0.1, test_ratio=0.1):
    train_parts = []
    val_parts = []
    test_parts = []

    for user_id, group in df.groupby("user_id"):
        group = group.sort_values("timestamp").copy()
        n = len(group)

        n_test = max(1, int(n * test_ratio))
        n_val = max(1, int(n * val_ratio))

        if n - n_val - n_test < 1:
            train_parts.append(group)
            continue

        train = group.iloc[: n - n_val - n_test]
        val = group.iloc[n - n_val - n_test : n - n_test]
        test = group.iloc[n - n_test :]

        train_parts.append(train)
        val_parts.append(val)
        test_parts.append(test)

    train_df = pd.concat(train_parts).reset_index(drop=True)
    val_df = pd.concat(val_parts).reset_index(drop=True)
    test_df = pd.concat(test_parts).reset_index(drop=True)

    return train_df, val_df, test_df

In [12]:
train_ratings, val_ratings, test_ratings = temporal_split(filtered_ratings)

In [13]:
assert len(train_ratings) + len(val_ratings) + len(test_ratings) == len(filtered_ratings)

In [14]:
print("Train shape:", train_ratings.shape)
print("Validation shape:", val_ratings.shape)
print("Test shape:", test_ratings.shape)

Train shape: (801390, 5)
Validation shape: (96882, 5)
Test shape: (96882, 5)


In [15]:
users_filtered = set(filtered_ratings["user_id"].unique())
users_train = set(train_ratings["user_id"].unique())
users_val = set(val_ratings["user_id"].unique())
users_test = set(test_ratings["user_id"].unique())

print("All users in train:", users_filtered == users_train)
print("All users in val:", users_filtered == users_val)
print("All users in test:", users_filtered == users_test)

All users in train: True
All users in val: True
All users in test: True


In [16]:
sample_user = 1

print("Train last date:",
      train_ratings[train_ratings["user_id"] == sample_user]["datetime"].max())

print("Val first date:",
      val_ratings[val_ratings["user_id"] == sample_user]["datetime"].min())

print("Test first date:",
      test_ratings[test_ratings["user_id"] == sample_user]["datetime"].min())

Train last date: 2001-01-06 23:37:48
Val first date: 2001-01-06 23:37:48
Test first date: 2001-01-06 23:38:11


### 5 Pairwise comparisons

## Convert rating data into pairwise preference data.
For each user, creates pairs (pos_movie, neg_movie) where the positive movie is rated at least 'min_rating_diff' higher than the negative movie.

Limits the number of pairs per user to 'max_pairs_per_user'.

In [17]:
def make_pairwise_data(
    df,
    min_rating_diff=1,
    max_pairs_per_user=200,
    random_state=42
):
    rng = np.random.default_rng(random_state)
    pair_rows = []

    for user_id, group in df.groupby("user_id"):
        user_movies = group[["movie_id", "rating"]].values.tolist()
        user_pairs = []

        for (movie_a, rating_a), (movie_b, rating_b) in combinations(user_movies, 2):
            diff = rating_a - rating_b

            if diff >= min_rating_diff:
                user_pairs.append((user_id, movie_a, movie_b, rating_a, rating_b, diff))
            elif diff <= -min_rating_diff:
                user_pairs.append((user_id, movie_b, movie_a, rating_b, rating_a, -diff))

        if len(user_pairs) > max_pairs_per_user:
            idx = rng.choice(len(user_pairs), size=max_pairs_per_user, replace=False)
            user_pairs = [user_pairs[i] for i in idx]

        pair_rows.extend(user_pairs)

    pairwise_df = pd.DataFrame(
        pair_rows,
        columns=["user_id", "pos_movie", "neg_movie", "pos_rating", "neg_rating", "rating_diff"]
    )

    return pairwise_df

In [18]:
train_pairs = make_pairwise_data(
    train_ratings,
    min_rating_diff=1,
    max_pairs_per_user=200,
    random_state=42
)

val_pairs = make_pairwise_data(
    val_ratings,
    min_rating_diff=1,
    max_pairs_per_user=200,
    random_state=42
)

test_pairs = make_pairwise_data(
    test_ratings,
    min_rating_diff=1,
    max_pairs_per_user=200,
    random_state=42
)

print("Train pairs shape:", train_pairs.shape)
print("Validation pairs shape:", val_pairs.shape)
print("Test pairs shape:", test_pairs.shape)

display(train_pairs.head())

Train pairs shape: (1155105, 6)
Validation pairs shape: (395462, 6)
Test pairs shape: (397534, 6)


,user_id,pos_movie,neg_movie,pos_rating,neg_rating,rating_diff
0,1,2018,661,4,3,1
1,1,1035,720,5,3,2
2,1,1193,661,5,3,2
3,1,527,531,5,4,1
4,1,1270,1197,5,3,2


In [19]:
print("All train pairs valid:", (train_pairs["pos_rating"] > train_pairs["neg_rating"]).all())
print("All val pairs valid:", (val_pairs["pos_rating"] > val_pairs["neg_rating"]).all())
print("All test pairs valid:", (test_pairs["pos_rating"] > test_pairs["neg_rating"]).all())

print("Min train rating diff:", train_pairs["rating_diff"].min())
print("Min val rating diff:", val_pairs["rating_diff"].min())
print("Min test rating diff:", test_pairs["rating_diff"].min())

print("Max train pairs/user:", train_pairs["user_id"].value_counts().max())
print("Max val pairs/user:", val_pairs["user_id"].value_counts().max())
print("Max test pairs/user:", test_pairs["user_id"].value_counts().max())

All train pairs valid: True
All val pairs valid: True
All test pairs valid: True
Min train rating diff: 1
Min val rating diff: 1
Min test rating diff: 1
Max train pairs/user: 200
Max val pairs/user: 200
Max test pairs/user: 200


In [20]:
train_users_without_pairs = set(train_ratings["user_id"].unique()) - set(train_pairs["user_id"].unique())
val_users_without_pairs = set(val_ratings["user_id"].unique()) - set(val_pairs["user_id"].unique())
test_users_without_pairs = set(test_ratings["user_id"].unique()) - set(test_pairs["user_id"].unique())

print("Users without train pairs:", len(train_users_without_pairs))
print("Users without val pairs:", len(val_users_without_pairs))
print("Users without test pairs:", len(test_users_without_pairs))

Users without train pairs: 4
Users without val pairs: 419
Users without test pairs: 421


### Save Results 

In [21]:
summary = {
    "original_ratings": len(ratings),
    "filtered_ratings": len(filtered_ratings),
    "unique_users_filtered": filtered_ratings["user_id"].nunique(),
    "unique_movies_filtered": filtered_ratings["movie_id"].nunique(),
    "min_ratings_per_user_after_filter": filtered_ratings["user_id"].value_counts().min(),
    "min_ratings_per_movie_after_filter": filtered_ratings["movie_id"].value_counts().min(),

    "train_ratings": len(train_ratings),
    "validation_ratings": len(val_ratings),
    "test_ratings": len(test_ratings),

    "users_in_train": train_ratings["user_id"].nunique(),
    "users_in_val": val_ratings["user_id"].nunique(),
    "users_in_test": test_ratings["user_id"].nunique(),

    "train_pairs": len(train_pairs),
    "validation_pairs": len(val_pairs),
    "test_pairs": len(test_pairs),

    "train_users_with_pairs": train_pairs["user_id"].nunique(),
    "val_users_with_pairs": val_pairs["user_id"].nunique(),
    "test_users_with_pairs": test_pairs["user_id"].nunique(),

    "min_train_rating_diff": train_pairs["rating_diff"].min(),
    "min_val_rating_diff": val_pairs["rating_diff"].min(),
    "min_test_rating_diff": test_pairs["rating_diff"].min(),

    "users_without_train_pairs": len(train_users_without_pairs),
    "users_without_val_pairs": len(val_users_without_pairs),
    "users_without_test_pairs": len(test_users_without_pairs),
}

summary_df = pd.DataFrame(summary.items(), columns=["metric", "value"])
display(summary_df)

,metric,value
0,original_ratings,1000209
1,filtered_ratings,995154
2,unique_users_filtered,6022
3,unique_movies_filtered,3043
4,min_ratings_per_user_after_filter,20
5,min_ratings_per_movie_after_filter,20
6,train_ratings,801390
7,validation_ratings,96882
8,test_ratings,96882
9,users_in_train,6022


In [22]:
import os

output_dir = "./outputs01"
os.makedirs(output_dir, exist_ok=True)

filtered_ratings.to_csv(f"{output_dir}/filtered_ratings.csv", index=False)
train_ratings.to_csv(f"{output_dir}/train_ratings.csv", index=False)
val_ratings.to_csv(f"{output_dir}/val_ratings.csv", index=False)
test_ratings.to_csv(f"{output_dir}/test_ratings.csv", index=False)

train_pairs.to_csv(f"{output_dir}/train_pairs.csv", index=False)
val_pairs.to_csv(f"{output_dir}/val_pairs.csv", index=False)
test_pairs.to_csv(f"{output_dir}/test_pairs.csv", index=False)

summary_df.to_csv(f"{output_dir}/step1_summary.csv", index=False)

print(f"All results saved to {output_dir}")

All results saved to ./outputs01
